# MORICHIKA Phase 2 — generalized LoRA hybrid v4
Same final full-coverage corpus/policy router as base v4, with one hash-bound private step-10 LoRA. Writes submission.csv; never submits.


In [ ]:
"""Context-only policy packet for the MORICHIKA generalized hybrid.

The module is intentionally self-contained so its exact source can be embedded
in an offline Kaggle notebook.  It never retrieves, labels, or consults a
competition row identifier.  It converts the supplied context, question and
candidate response into an auditable instruction packet for the verifier.
"""

from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from typing import Any, Callable


VERSION = "morichika-context-policy-v4-full-coverage-map-aggregate"

# These are reasoning checks, not predicted classes.  Keeping the recovered
# inventory explicit prevents a later prompt edit from silently dropping an
# edge-case family.
CANONICAL_POLICY_FAMILIES = (
    "question_grounding_answerability_and_premise_validity",
    "exact_entity_relation_property_and_answer_type",
    "direct_support_contradiction_and_partial_containment",
    "supplied_definition_formula_theory_and_rule_application",
    "date_event_role_and_as_of_time",
    "year_age_duration_calendar_and_timezone",
    "bounded_arithmetic_units_ratio_percentage_and_order",
    "kinship_and_relational_composition",
    "creator_founder_user_operator_office_title_and_jurisdiction",
    "birthplace_residence_nationality_and_event_participation",
    "legal_definition_section_effective_date_minimum_maximum_and_frequency",
    "numeric_whole_component_range_extremum_ordinal_and_granularity",
    "negation_quantifier_comparator_modality_and_clause_scope",
    "antonym_idiom_prefix_register_etymology_and_guruchandali",
    "samas_sandhi_affix_spelling_natva_and_satva",
    "unicode_joiner_conjunct_digit_punctuation_ocr_and_word_break",
    "ambiguity_conflict_invalid_premise_and_no_world_rescue",
)

ENGINEERED_EVALUATION_CELLS = (
    "context_only_lane", "full_context_boundary", "answerable_supported",
    "answerable_refuted", "genuinely_missing_nei", "same_passage_different_question",
    "entity_swap", "relation_swap", "answer_type_swap", "creator_vs_user_operator",
    "birthplace_vs_residence_event_place", "partial_answer_extra_claim",
    "theory_application", "formula_operand_application", "age_vs_year",
    "relative_timeline", "event_phase_order", "kinship_composition",
    "unit_or_number_swap", "negation_scope", "quantifier_comparator_scope",
    "grammar_operation_swap", "lexical_exact_vs_semantic_near",
    "unicode_equivalent", "unicode_word_break_non_equivalent", "cross_window_conflict",
)

OPERATION_AXIS = (
    "antonym_lookup", "birth_date_slot", "entity_text_relation",
    "event_date_or_status_slot", "idiom_meaning_lookup",
    "legal_definition_or_threshold", "legal_effective_date", "legal_maximum_fine",
    "legal_maximum_imprisonment", "legal_minimum_meeting_frequency",
    "location_slot", "numeric_or_ordinal_slot", "prefix_origin_classification",
    "samas_taxonomy", "sandhi_formation",
)

_SIGNALS: tuple[tuple[str, str], ...] = (
    ("year_age_duration", r"ব[য়য়]স|জন্ম|বিবাহ|বিয়ে|বিয়ে|বছর|সাল|year|age"),
    ("calendar_date", r"তারিখ|কবে|দিন|মাস|জানুয়ারি|ফেব্রুয়ারি|মার্চ|ডিসেম্বর"),
    ("timeline_event_order", r"আগে|পরে|তারপর|পরবর্তীতে|প্রথমে|শেষে|ঘটে|ঘটেছিল"),
    ("relative_time_offset", r"(?:বছর|দিন|মাস)\s*(?:আগে|পরে)|after|before|later"),
    ("kinship_relation_composition", r"বাবা|পিতা|মা|মাতা|দাদা|পিতামহ|নানা|grandfather"),
    ("bounded_arithmetic", r"কত|যোগ|বিয়োগ|বিয়োগ|গুণ|ভাগ|শতাংশ|হিসাব|calculate"),
    ("unit_dimension", r"কিলোমিটার|মিটার|গ্রাম|লিটার|সেকেন্ড|মিনিট|ঘণ্টা|টাকা|শতাংশ"),
    ("negation_scope", r"(?:^|\s)(?:না|নয়|নয়|নেই|নি)(?:\s|$)|হয়নি|হয়নি|করেনি"),
    ("quantifier_scope", r"সব|সকল|কোনো|কোনও|কেবল|শুধু|প্রত্যেক|একটিও|কিছু"),
    ("comparator_scope", r"সর্বাধিক|সর্বনিম্ন|বেশি|কম|পূর্ববর্তী|পরবর্তী|minimum|maximum"),
    ("modality_clause_scope", r"পারে|উচিত|অবশ্যই|সম্ভব|শর্তে|যদি|হলে"),
    ("definition_theory_rule_application", r"সংজ্ঞা|তত্ত্ব|সূত্র|নিয়ম|নিয়ম|অর্থ"),
    ("antonym_exact_pair", r"বিপরীত\s*শব্দ|antonym"),
    ("idiom_exact_meaning", r"বাগধারা|প্রবাদ"),
    ("prefix_suffix_origin", r"উপসর্গ|প্রত্যয়|প্রত্যয়"),
    ("samas_exact_operation", r"সমাস"),
    ("sandhi_exact_operands", r"সন্ধি"),
    ("register_etymology_guruchandali", r"গুরুচণ্ডালী|তৎসম|তদ্ভব|ফারসি|সংস্কৃত"),
    ("natva_satva_spelling", r"ণত্ব|ষত্ব|বানান"),
    ("creator_user_operator_role", r"স্রষ্টা|প্রতিষ্ঠাতা|নির্মাতা|ব্যবহারকারী|পরিচালক"),
    ("birthplace_residence_event_place", r"জন্মস্থান|বাসস্থান|রাজধানী|কোথায়|কোথায়|স্থান"),
    ("legal_definition_section", r"আইন|ধারা|বিধি|সংজ্ঞা"),
    ("minimum_maximum_frequency", r"সর্বনিম্ন|সর্বোচ্চ|কমপক্ষে|বেশিরভাগ|বার"),
    ("ordinal_list_order", r"প্রথম|দ্বিতীয়|দ্বিতীয়|তৃতীয়|তৃতীয়|ক্রম|তম"),
)

_BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
_JOINERS = {"\u200c", "\u200d", "\ufeff"}
_FIXED_LEXICAL_SHELLS = {
    "antonym_lookup": re.compile(r"(?:বিপরীত\s*শব্দ|বিপরীতার্থক\s*শব্দ).*(?:কী|কি|কোন)", re.IGNORECASE),
    "idiom_meaning_lookup": re.compile(r"(?:বাগধারা|প্রবাদ).*(?:অর্থ|মানে).*(?:কী|কি|কোন)", re.IGNORECASE),
    "prefix_origin_classification": re.compile(r"উপসর্গ.*(?:উৎস|শ্রেণি|জাতীয়|জাতীয়|কোন)", re.IGNORECASE),
}


def _sha(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def _canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def comparison_view(value: str) -> str:
    """Bounded comparison only; never rewrites literal evidence."""
    return " ".join(unicodedata.normalize("NFC", value).translate(_BN_DIGITS).casefold().split())


def unicode_receipt(value: str) -> dict[str, Any]:
    nfc = unicodedata.normalize("NFC", value)
    return {
        "literal_sha256": _sha(value),
        "utf8_bytes": len(value.encode("utf-8")),
        "codepoints": len(value),
        "nfc_identical": nfc == value,
        "nfc_sha256": _sha(nfc),
        "joiner_positions": [index for index, char in enumerate(value) if char in _JOINERS],
        "replacement_character_positions": [index for index, char in enumerate(value) if char == "\ufffd"],
        "comparison_view_sha256": _sha(comparison_view(value)),
    }


def contextual_exact_lexical_policy(
    question: str, response: str, records: list[dict[str, Any]] | None
) -> dict[str, Any]:
    """Expose only the frozen exact lexical-shell exception, nonterminally."""
    folded_question = comparison_view(question)
    folded_response = comparison_view(response)
    operation = next(
        (name for name, pattern in _FIXED_LEXICAL_SHELLS.items() if pattern.search(folded_question)),
        None,
    )
    base = {
        "lookup_mode": "not_applicable",
        "operation": operation,
        "key": None,
        "source_ids": [],
        "record_sha256": [],
        "exact_operation": operation is not None,
        "exact_key": False,
        "exact_sense_register": False,
        "conflict": False,
        "conflict_status": "none",
        "generic_retrieval_invoked": False,
        "terminal_label_authority": False,
        "model_nonterminal": True,
        "evidence": [],
    }
    if operation is None:
        base["lookup_mode"] = "forbidden_for_ordinary_context"
        return base

    matched: list[dict[str, Any]] = []
    conflicting: list[dict[str, Any]] = []
    for record in records or []:
        if record.get("operation") != operation:
            continue
        terms = [str(record.get("term_key") or "")] + [str(v) for v in record.get("display_terms", [])]
        exact_terms = []
        for term in terms:
            key = comparison_view(term)
            if key and re.search(rf"(?<![\u0980-\u09FFA-Za-z0-9]){re.escape(key)}(?![\u0980-\u09FFA-Za-z0-9])", folded_question):
                exact_terms.append(key)
        if not exact_terms:
            continue
        sense_values = [str(record.get(name) or "") for name in ("sense", "register", "etymological_class")]
        required_scope = [comparison_view(value) for value in sense_values if value]
        exact_scope = not required_scope or all(value in folded_question for value in required_scope)
        candidate = {**record, "_matched_key": sorted(exact_terms, key=lambda value: (-len(value), value))[0], "_exact_scope": exact_scope}
        if record.get("conflict_status") == "none" and exact_scope:
            matched.append(candidate)
        else:
            conflicting.append(candidate)

    if not matched and not conflicting:
        base["lookup_mode"] = "exact_shell_no_exact_key_nonterminal"
        return base
    all_candidates = matched + conflicting
    keys = {value["_matched_key"] for value in all_candidates}
    accepted_sets = {
        tuple(sorted(comparison_view(v) for v in value.get("accepted_answers", []) if str(v).strip()))
        for value in matched
    }
    conflict = bool(conflicting) or len(keys) != 1 or len(accepted_sets) > 1
    base.update({
        "lookup_mode": "exact_hash_bound_lexical_policy" if not conflict else "exact_lexical_conflict_nei_nonterminal",
        "key": sorted(keys)[0] if len(keys) == 1 else None,
        "source_ids": sorted({str(value.get("source_id") or "") for value in all_candidates if value.get("source_id")}),
        "record_sha256": sorted({_sha(_canonical({k: v for k, v in value.items() if not str(k).startswith("_")})) for value in all_candidates}),
        "exact_key": len(keys) == 1,
        "exact_sense_register": all(value.get("_exact_scope") is True for value in all_candidates),
        "conflict": conflict,
        "conflict_status": "conflict" if conflict else "none",
    })
    if not conflict:
        for value in matched[:3]:
            accepted = [comparison_view(v) for v in value.get("accepted_answers", [])]
            contrast = [comparison_view(v) for v in value.get("contrast_answers", [])]
            base["evidence"].append({
                "source_id": value.get("source_id"),
                "operation": operation,
                "term_key": value.get("term_key"),
                "accepted_answers": value.get("accepted_answers", []),
                "contrast_answers": value.get("contrast_answers", []),
                "response_relation": (
                    "support_candidate" if folded_response in accepted else
                    "counter_candidate" if folded_response in contrast else "neutral_candidate"
                ),
            })
    return base


def detected_policy_families(context: str, question: str, response: str) -> list[str]:
    joined = "\n".join((question, response, context))
    detected = {
        "lane_context_only", "full_context_evidence_boundary", "question_answerability",
        "question_premise_validity", "exact_answer_slot", "same_passage_different_question",
        "entity_identity", "relation_property_identity", "answer_type_identity",
        "direct_support", "direct_contradiction", "partial_containment_extra_claim",
        "ambiguity_conflict_nei", "unicode_nfc_joiner_conjunct_digit", "ocr_word_break_caution",
    }
    for family, pattern in _SIGNALS:
        if re.search(pattern, joined, re.IGNORECASE):
            detected.add(family)
    if re.search(r"[০-৯0-9]", joined):
        detected.update({"number_whole_component_range", "formula_operand_application"})
    order = [name for name, _ in _SIGNALS]
    order += [
        "lane_context_only", "full_context_evidence_boundary", "question_answerability",
        "question_premise_validity", "exact_answer_slot", "same_passage_different_question",
        "entity_identity", "relation_property_identity", "answer_type_identity",
        "direct_support", "direct_contradiction", "partial_containment_extra_claim",
        "ambiguity_conflict_nei", "unicode_nfc_joiner_conjunct_digit", "ocr_word_break_caution",
        "number_whole_component_range", "formula_operand_application",
    ]
    return [family for family in dict.fromkeys(order) if family in detected]


def full_coverage_windows(
    context: str, *, max_chars: int = 3000, overlap_chars: int = 320
) -> list[dict[str, Any]]:
    """Partition every context character into overlapping replayable windows."""
    if not context:
        raise ValueError("context must be non-empty")
    if max_chars < 256 or overlap_chars < 32 or overlap_chars >= max_chars:
        raise ValueError("invalid full-coverage window geometry")
    windows: list[dict[str, Any]] = []
    start = 0
    while start < len(context):
        hard_end = min(len(context), start + max_chars)
        end = hard_end
        if hard_end < len(context):
            boundaries = [context.rfind(mark, start + max_chars // 2, hard_end) for mark in ("\n", "।", ".", "?", "!")]
            boundary = max(boundaries)
            if boundary >= start:
                end = boundary + 1
        literal = context[start:end]
        windows.append({
            "window_index": len(windows),
            "context_char_start": start,
            "context_char_end": end,
            "literal_text": literal,
            "literal_span_sha256": _sha(literal),
        })
        if end == len(context):
            break
        start = end - overlap_chars
    covered = [False] * len(context)
    for index, window in enumerate(windows):
        start, end = window["context_char_start"], window["context_char_end"]
        literal = window["literal_text"]
        if context[start:end] != literal or _sha(literal) != window["literal_span_sha256"]:
            raise ValueError(f"full-coverage window {index} does not replay")
        for position in range(start, end):
            covered[position] = True
    if not all(covered):
        raise ValueError("full-coverage window ledger contains a character gap")
    return windows


def build_window_adjudication_user(
    window: dict[str, Any], question: str, response: str, total_windows: int
) -> str:
    return (
        "WINDOW-LOCAL CONTEXT PASS. Judge only evidence present in this literal supplied-context window. "
        "A local miss is not a contradiction: output not_enough_information unless this window directly "
        "supports or directly refutes the exact question slot. Do not use outside knowledge.\n\n"
        f"QUESTION (literal):\n{question}\n\nCANDIDATE RESPONSE (literal):\n{response}\n\n"
        f"WINDOW {window['window_index'] + 1}/{total_windows} "
        f"[chars={window['context_char_start']}:{window['context_char_end']}; "
        f"sha256={window['literal_span_sha256']}]:\n{window['literal_text']}"
    )


def _literal_question_excerpt(
    window: dict[str, Any], question: str, max_chars: int
) -> dict[str, Any]:
    """Select a literal, question-conditioned excerpt without rewriting it."""
    literal = str(window["literal_text"])
    if max_chars < 32:
        max_chars = 32
    q_tokens = {
        token.casefold() for token in re.findall(r"[\u0980-\u09FFA-Za-z0-9]+", question)
        if len(token) > 1
    }
    candidates = list(re.finditer(r"[^।.!?\n]+[।.!?]?", literal))
    if candidates:
        def score(match: re.Match[str]) -> tuple[int, int]:
            tokens = {
                token.casefold() for token in re.findall(r"[\u0980-\u09FFA-Za-z0-9]+", match.group())
            }
            return (len(tokens & q_tokens), -match.start())
        best = max(candidates, key=score)
        local_start = max(0, best.start() - max_chars // 4)
        local_end = min(len(literal), local_start + max_chars)
    else:
        local_start, local_end = 0, min(len(literal), max_chars)
    excerpt = literal[local_start:local_end]
    absolute_start = int(window["context_char_start"]) + local_start
    absolute_end = absolute_start + len(excerpt)
    return {
        "excerpt_char_start": absolute_start,
        "excerpt_char_end": absolute_end,
        "literal_excerpt": excerpt,
        "literal_excerpt_sha256": _sha(excerpt),
    }


def build_aggregation_user(
    question: str,
    response: str,
    window_results: list[dict[str, Any]],
    *,
    selected_notes: list[dict[str, Any]] | None = None,
    bounded_derivations: list[dict[str, Any]] | None = None,
    lexical_policy: dict[str, Any] | None = None,
) -> str:
    """Create the exact-question final adjudication over all window passes."""
    compact = [
        {
            "window_index": row["window_index"],
            "context_char_start": row["context_char_start"],
            "context_char_end": row["context_char_end"],
            "literal_span_sha256": row["literal_span_sha256"],
            "window_verdict": row["window_verdict"],
            "literal_excerpt": row.get("literal_excerpt", ""),
            "excerpt_char_start": row.get("excerpt_char_start"),
            "excerpt_char_end": row.get("excerpt_char_end"),
            "literal_excerpt_sha256": row.get("literal_excerpt_sha256"),
        }
        for row in window_results
    ]
    notes = list(selected_notes or [])[:32]
    derivations = list(bounded_derivations or [])[:8]
    return (
        "FINAL FULL-CONTEXT AGGREGATION. The window ledger collectively covered every supplied-context "
        "character. Rebind the exact question and candidate response. A not_enough_information window means "
        "only that its local span was silent; it is never counter-evidence. If any aligned window refutes the "
        "response, do not let an unrelated support override it. Unresolved support/refutation across different "
        "entities, slots, event phases, dates, operations, or ambiguous coreference is not_enough_information. "
        "Use no outside knowledge.\n\n"
        f"QUESTION (literal):\n{question}\n\nCANDIDATE RESPONSE (literal):\n{response}\n\n"
        "PER-WINDOW STRUCTURED RESULTS AND LITERAL EXCERPTS:\n" + _canonical(compact)
        + "\n\nFULL-CONTEXT ROUTER SOURCE-LINKED NOTES (advisory; offsets/hashes bind them to supplied context):\n"
        + _canonical(notes)
        + "\n\nBOUNDED DERIVATION CANDIDATES (advisory; verify operator, operands, slot and unit):\n"
        + _canonical(derivations)
        + "\n\nFROZEN EXACT LEXICAL-SHELL POLICY (nonterminal; ordinary retrieval forbidden):\n"
        + _canonical(lexical_policy or {"lookup_mode": "not_applicable", "generic_retrieval_invoked": False})
    )


def build_contextual_policy_packet(
    context: str,
    question: str,
    response: str,
    router: Callable[..., dict[str, Any]],
    lexical_records: list[dict[str, Any]] | None = None,
    *,
    max_windows: int = 8,
    full_context_char_limit: int = 6000,
) -> tuple[str, dict[str, Any]]:
    """Build a context-only prompt plus restart-safe diagnostics."""
    if not all(isinstance(value, str) for value in (context, question, response)):
        raise TypeError("context, question and response must be strings")
    if not context.strip() or not question.strip():
        raise ValueError("context and question must be non-empty")

    route = router(context, question, response, max_windows=max_windows)
    if route.get("external_retrieval_allowed") is not False:
        raise ValueError("context router must explicitly forbid external retrieval")
    if route.get("context_sha256") != _sha(context):
        raise ValueError("context router hash mismatch")

    window_calls: list[dict[str, Any]] = []
    if len(context) <= full_context_char_limit:
        evidence = context
        evidence_mode = "complete_literal_supplied_context"
        full_context_inference_coverage = True
    else:
        coverage = full_coverage_windows(context)
        excerpt_chars = max(48, min(320, 3200 // len(coverage)))
        window_calls = [
            {
                **{key: window[key] for key in (
                    "window_index", "context_char_start", "context_char_end", "literal_span_sha256"
                )},
                **_literal_question_excerpt(window, question, excerpt_chars),
                "user": build_window_adjudication_user(window, question, response, len(coverage)),
            }
            for window in coverage
        ]
        # The ordinary single-call payload is never used for a long context;
        # it is deliberately tiny so no truncated projection can be mistaken
        # for the evidence universe.
        evidence = "[FULL-COVERAGE WINDOW PASSES REQUIRED BEFORE AGGREGATION]"
        evidence_mode = "all_literal_windows_then_final_aggregation"
        full_context_inference_coverage = True

    signals = detected_policy_families(context, question, response)
    lexical_policy = contextual_exact_lexical_policy(question, response, lexical_records)
    aggregation_derivations = [
        value for value in list(route.get("bounded_derivation_candidates") or [])[:8]
        if isinstance(value, dict) and value.get("source_linked") is True
    ]
    operand_ids = {
        str(note_id)
        for value in aggregation_derivations
        for note_id in value.get("operand_note_ids", [])
    }
    routed_notes = list(route.get("selected_notes") or [])
    routed_notes.sort(key=lambda note: (str(note.get("note_id")) not in operand_ids, int(note.get("context_char_start", 0))))
    aggregation_notes = []
    note_budget = 2400
    used_note_chars = 0
    for note in routed_notes[:32]:
        start = int(note["context_char_start"])
        end = int(note["context_char_end"])
        literal = str(note["literal_text"])
        if context[start:end] != literal or _sha(literal) != note["literal_span_sha256"]:
            raise ValueError("router selected note does not replay against supplied context")
        serialized_chars = len(_canonical(note))
        if aggregation_notes and used_note_chars + serialized_chars > note_budget:
            continue
        aggregation_notes.append(note)
        used_note_chars += serialized_chars
    notes = {
        "selected_notes": aggregation_notes,
        "bounded_derivation_candidates": aggregation_derivations,
    }
    contract = {
        "version": VERSION,
        "evidence_universe": "only_the_literal_supplied_context",
        "external_retrieval_allowed": False,
        "outside_world_fact_rescue_allowed": False,
        "evidence_mode": evidence_mode,
        "full_context_inference_coverage": full_context_inference_coverage,
        "every_context_character_processed": True,
        "detected_policy_families": signals,
        "checks_in_order": [
            "validate question premise and whether the requested slot is answerable",
            "bind exact entity relation property answer type event phase time polarity comparator and unit",
            "seek direct support and direct contradiction for that exact slot",
            "apply only supplied definition theory rule formula and bounded operands",
            "separate refuted from missing ambiguous conflicting or locally silent window evidence",
            "compare Unicode only through bounded NFC digit joiner and attested word-break views; near-looking forms are not automatically equal",
        ],
        "verdict_rule": {
            "supported": "the complete response follows for the exact requested slot",
            "refuted": "the supplied context establishes an incompatible value or claim",
            "not_enough_information": "required evidence is absent omitted ambiguous conflicting or premise-invalid",
        },
        "fixed_lexical_exception": lexical_policy,
    }
    user = (
        "CONTEXT-ONLY VERIFICATION CONTRACT:\n" + _canonical(contract)
        + "\n\nQUESTION (literal):\n" + question
        + "\n\nSUPPLIED CONTEXT EVIDENCE (literal; this is the only evidence universe):\n" + evidence
        + "\n\nSOURCE-LINKED EXTRACTIVE NOTES (advisory, never new evidence):\n" + _canonical(notes)
        + "\n\nCANDIDATE RESPONSE (verify every material claim):\n" + response
    )
    diagnostic = {
        **route,
        "context_policy_version": VERSION,
        "policy_family_inventory_count": len(CANONICAL_POLICY_FAMILIES),
        "canonical_policy_family_count": len(CANONICAL_POLICY_FAMILIES),
        "engineered_evaluation_cell_count": len(ENGINEERED_EVALUATION_CELLS),
        "operation_axis_count": len(OPERATION_AXIS),
        "canonical_policy_families": list(CANONICAL_POLICY_FAMILIES),
        "engineered_evaluation_cells": list(ENGINEERED_EVALUATION_CELLS),
        "operation_axis": list(OPERATION_AXIS),
        "detected_policy_families": signals,
        "evidence_mode": evidence_mode,
        "full_context_inference_coverage": full_context_inference_coverage,
        "context_literal": unicode_receipt(context),
        "question_literal": unicode_receipt(question),
        "response_literal": unicode_receipt(response),
        "prompt_sha256": _sha(user),
        "requires_window_aggregation": bool(window_calls),
        "window_calls": window_calls,
        "window_count": len(window_calls) if window_calls else 1,
        "full_context_character_count": len(context),
        "aggregation_selected_notes": aggregation_notes,
        "aggregation_bounded_derivations": aggregation_derivations,
        "contextual_lexical_policy": lexical_policy,
    }
    diagnostic.pop("receipt_sha256", None)
    diagnostic["receipt_sha256"] = _sha(_canonical(diagnostic))
    return user, diagnostic


__all__ = [
    "VERSION", "CANONICAL_POLICY_FAMILIES", "ENGINEERED_EVALUATION_CELLS",
    "OPERATION_AXIS", "comparison_view", "unicode_receipt",
    "detected_policy_families", "full_coverage_windows",
    "build_window_adjudication_user", "build_aggregation_user",
    "build_contextual_policy_packet",
]


In [ ]:
"""BICHAR Phase-2 private offline T4x2 base-Gemma heavy hybrid runner."""

from __future__ import annotations

import concurrent.futures
import csv
import hashlib
import json
import os
import re
import shutil
import stat
import subprocess
import sys
import time
import unicodedata
import zipfile
from collections import Counter
from pathlib import Path, PurePosixPath
from typing import Any

import pandas as pd
import requests


VERSION = "morichika-phase2-generalized-lora-hybrid-v4"
INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
RUN_ROOT = WORKING_ROOT / "morichika_phase2_generalized_lora_v4"
MODEL_FILE = "gemma-4-E4B-it-qat-UD-Q4_K_XL.gguf"
MODEL_BYTES = 4_215_695_776
MODEL_SHA256 = "df0fd4ee07072c607c29a0a1cb4f98918426cca12f45a2776bdd6ee6d09a4de3"
ADAPTER_DATASET_ID = 'ishtyy/bichar-gemma4-e4b-step10-lora-gguf-20260720'
ADAPTER_FILE = 'gemma4-e4b-lr2e6-verdict45-step10-f16.gguf'
ADAPTER_BYTES = 69799104
ADAPTER_SHA256 = 'ebef35b50d943ccb94a0749993b20aa408331d0615a2639525be263cdfe8a8e2'
ADAPTER_MANIFEST_ID = '67b234323047dbf932f43133ccde5d1de720aea8f15b418a32fa90101ec9ee0b'
ADAPTER_CONVERSION_ID = '2ee373a1aff488777a3442a2ec0d085955b14726db4439987f0ad95954313526'
ADAPTER_RIGHTS_ID = '875e4b7eeae22f3266507fb202574f3adcf7612070f439fac44db2d36e00d897'
LLAMA_COMMIT = "86d86ed4396b4130922f7b9af26e3d9fc11a591b"
RUNTIME_FILE = f"llama_cpp_{LLAMA_COMMIT}_cuda128_sm75.bin"
RUNTIME_BYTES = 102_845_584
RUNTIME_SHA256 = "1c75bfb81d49cd99644696381e9ab63c19baa6c2cc099b258827a3eb8a16846d"
RUNTIME_MANIFEST_ID = "a80fa04de769a2d77c26a862a640fb6eae35101929c89d649cb5748faaf2b6ea"
RETRIEVAL_DATASET_ID = 'ishtyy/morichika-phase2-retrieval-strict-v3-20260720'
RETRIEVAL_MANIFEST_ID = 'bfd6eecb3d011462d45282aa79967cd5258a90c6c5324193b5bb28d5a3c439b8'
RETRIEVAL_ARCHIVES = {
    "artifacts": "artifacts.zip",
    "pipeline": "pipeline.zip",
    "source_registry_v6": "source_registry_v6.zip",
}
BASE_SEED = 7_170_064
MAX_NEW_TOKENS = 16
DEADLINE_SECONDS = 8 * 60 * 60 + 30 * 60
NULL_CONTEXTS = {"", "closed_book", "[null]", "null", "nan", "none", "n/a"}
VERDICT_ONLY_LITERALS = {
    '{"verdict":"supported"}': 1,
    '{"verdict":"refuted"}': 0,
    '{"verdict":"not_enough_information"}': 0,
}
VERDICT_ONLY_GBNF = (
    'root ::= "{\\"verdict\\":\\\"" verdict "\\\"}"\n'
    'verdict ::= "supported" | "refuted" | "not_enough_information"'
)


class HybridFailure(RuntimeError):
    pass


def canonical(value: object) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def sha256_file(path: Path) -> str:
    value = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            value.update(chunk)
    return value.hexdigest()


def digest(value: object) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def atomic_text(path: Path, value: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + f".{os.getpid()}.tmp")
    temporary.write_text(value, encoding="utf-8", newline="\n")
    os.replace(temporary, path)


def atomic_json(path: Path, value: object) -> None:
    atomic_text(path, json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + "\n")


def read_json(path: Path) -> dict[str, Any]:
    value = json.loads(path.read_text(encoding="utf-8-sig"))
    if not isinstance(value, dict):
        raise HybridFailure(f"expected object: {path}")
    return value


def run(command: list[str], *, timeout: int = 1_800, env: dict[str, str] | None = None) -> subprocess.CompletedProcess[str]:
    result = subprocess.run(
        command, capture_output=True, text=True, shell=False, timeout=timeout,
        env=env, encoding="utf-8", errors="replace", check=False,
    )
    if result.returncode != 0:
        raise HybridFailure(f"command_failed:{Path(command[0]).name}:{result.returncode}:{(result.stderr or result.stdout)[-3000:]}")
    return result


def resolve_bound_file(name: str, size: int, expected_sha256: str) -> Path:
    matches = [
        path for path in INPUT_ROOT.rglob(name)
        if path.is_file() and not path.is_symlink() and path.stat().st_size == size
    ]
    valid = [path for path in matches if sha256_file(path) == expected_sha256]
    if not valid:
        raise HybridFailure(f"bound_file_missing:{name}")
    return sorted(valid, key=lambda path: (len(path.parts), str(path)))[0]


def _suffix_match(path: Path, relative: PurePosixPath) -> bool:
    parts = tuple(path.parts)
    wanted = tuple(relative.parts)
    return len(parts) >= len(wanted) and parts[-len(wanted):] == wanted


def _safe_archive_map(
    archive_path: Path,
    prefix: str,
    expected_paths: set[str],
) -> dict[str, zipfile.ZipInfo]:
    """Bind every regular ZIP member to one declared manifest path."""
    mapped: dict[str, zipfile.ZipInfo] = {}
    seen_names: set[str] = set()
    with zipfile.ZipFile(archive_path) as archive:
        for info in archive.infolist():
            name = info.filename
            path = PurePosixPath(name)
            mode = (info.external_attr >> 16) & 0xFFFF
            if (
                name in seen_names or path.is_absolute() or ".." in path.parts
                or "\\" in name or info.flag_bits & 1 or stat.S_ISLNK(mode)
            ):
                raise HybridFailure(f"unsafe_retrieval_archive_member:{archive_path.name}:{name}")
            seen_names.add(name)
            if info.is_dir():
                continue
            candidates = [path.as_posix(), (PurePosixPath(prefix) / path).as_posix()]
            declared = [candidate for candidate in candidates if candidate in expected_paths]
            if len(set(declared)) != 1:
                raise HybridFailure(f"undeclared_or_ambiguous_retrieval_archive_member:{archive_path.name}:{name}")
            relative = declared[0]
            if relative in mapped:
                raise HybridFailure(f"duplicate_retrieval_archive_payload:{relative}")
            mapped[relative] = info
    if set(mapped) != expected_paths:
        missing = sorted(expected_paths - set(mapped))
        raise HybridFailure(f"retrieval_archive_incomplete:{archive_path.name}:{missing[:3]}")
    return mapped


def _copy_archive_member(
    archive_path: Path,
    info: zipfile.ZipInfo,
    destination: Path,
    expected_bytes: int,
    expected_sha256: str,
) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + f".{os.getpid()}.tmp")
    value = hashlib.sha256()
    observed = 0
    try:
        with zipfile.ZipFile(archive_path) as archive, archive.open(info, "r") as source, temporary.open("wb") as target:
            while True:
                chunk = source.read(8 * 1024 * 1024)
                if not chunk:
                    break
                observed += len(chunk)
                value.update(chunk)
                target.write(chunk)
        if observed != expected_bytes or value.hexdigest() != expected_sha256:
            raise HybridFailure(f"retrieval_archive_member_binding_failed:{info.filename}")
        os.replace(temporary, destination)
    finally:
        if temporary.exists():
            temporary.unlink()


def _materialize_bound_payload_zip() -> tuple[Path, dict[str, Any]] | None:
    """Safely expand the one manifest-bound payload.zip Kaggle may expose."""
    matches = []
    for archive_path in INPUT_ROOT.rglob("payload.zip"):
        if not archive_path.is_file() or archive_path.is_symlink():
            continue
        try:
            with zipfile.ZipFile(archive_path) as archive:
                infos = archive.infolist()
                if len({info.filename for info in infos}) != len(infos):
                    raise HybridFailure("duplicate_monolithic_retrieval_member")
                manifest_infos = []
                for info in infos:
                    path = PurePosixPath(info.filename)
                    mode = (info.external_attr >> 16) & 0xFFFF
                    if (
                        path.is_absolute() or ".." in path.parts or "\\" in info.filename
                        or info.flag_bits & 1 or stat.S_ISLNK(mode)
                    ):
                        raise HybridFailure(f"unsafe_monolithic_retrieval_member:{info.filename}")
                    if not info.is_dir() and path.name == "bundle_manifest.json":
                        if info.file_size > 2 * 1024 * 1024:
                            raise HybridFailure("retrieval_bundle_manifest_oversized")
                        manifest_infos.append(info)
                for info in manifest_infos:
                    value = json.loads(archive.read(info).decode("utf-8-sig"))
                    if (
                        isinstance(value, dict)
                        and value.get("dataset_id") == RETRIEVAL_DATASET_ID
                        and value.get("manifest_id") == RETRIEVAL_MANIFEST_ID
                    ):
                        matches.append((archive_path, info, value))
        except (zipfile.BadZipFile, UnicodeDecodeError, json.JSONDecodeError):
            continue
    if not matches:
        return None
    if len(matches) != 1:
        raise HybridFailure(f"monolithic_retrieval_bundle_not_unique:{len(matches)}")
    archive_path, manifest_info, manifest = matches[0]
    core = {key: value for key, value in manifest.items() if key != "manifest_id"}
    if digest(core) != RETRIEVAL_MANIFEST_ID:
        raise HybridFailure("private_retrieval_manifest_identity_mismatch")
    specs = {str(spec["path"]): spec for spec in manifest.get("files", [])}
    prefix = PurePosixPath(manifest_info.filename).parent
    mapped = {}
    with zipfile.ZipFile(archive_path) as archive:
        for info in archive.infolist():
            if info.is_dir():
                continue
            path = PurePosixPath(info.filename)
            if prefix.parts:
                if tuple(path.parts[:len(prefix.parts)]) != tuple(prefix.parts):
                    raise HybridFailure(f"monolithic_retrieval_mixed_prefix:{info.filename}")
                relative = PurePosixPath(*path.parts[len(prefix.parts):]).as_posix()
            else:
                relative = path.as_posix()
            if relative == "bundle_manifest.json":
                continue
            if relative not in specs or relative in mapped:
                raise HybridFailure(f"undeclared_or_duplicate_monolithic_member:{relative}")
            mapped[relative] = info
    if set(mapped) != set(specs):
        raise HybridFailure("monolithic_retrieval_archive_incomplete")
    destination = RUN_ROOT / "payload_zip_input"
    destination.mkdir(parents=True, exist_ok=True)
    atomic_json(destination / "bundle_manifest.json", manifest)
    for relative, info in mapped.items():
        spec = specs[relative]
        if info.file_size != int(spec["bytes"]):
            raise HybridFailure(f"retrieval_archive_member_size_mismatch:{relative}")
        _copy_archive_member(
            archive_path, info, destination.joinpath(*PurePosixPath(relative).parts),
            int(spec["bytes"]), str(spec["sha256"]),
        )
    return destination / "bundle_manifest.json", manifest


def materialize_retrieval_payload() -> tuple[Path, dict[str, Any]]:
    manifests = []
    for path in INPUT_ROOT.rglob("bundle_manifest.json"):
        try:
            value = read_json(path)
        except Exception:
            continue
        if value.get("dataset_id") == RETRIEVAL_DATASET_ID and value.get("manifest_id") == RETRIEVAL_MANIFEST_ID:
            manifests.append((path, value))
    if not manifests:
        materialized = _materialize_bound_payload_zip()
        if materialized is None:
            raise HybridFailure("private_retrieval_bundle_missing")
        manifests.append(materialized)
    manifest_path, manifest = sorted(manifests, key=lambda item: (len(item[0].parts), str(item[0])))[0]
    manifest_core = {key: value for key, value in manifest.items() if key != "manifest_id"}
    if digest(manifest_core) != RETRIEVAL_MANIFEST_ID:
        raise HybridFailure("private_retrieval_manifest_identity_mismatch")
    stage = RUN_ROOT / "retrieval_payload"
    stage.mkdir(parents=True, exist_ok=True)
    input_files = [path for path in manifest_path.parent.rglob("*") if path.is_file() and not path.is_symlink()]
    specs = {str(spec["path"]): spec for spec in manifest.get("files", [])}
    archive_members: dict[str, tuple[Path, zipfile.ZipInfo]] = {}
    for prefix, archive_name in RETRIEVAL_ARCHIVES.items():
        expected = {name for name in specs if PurePosixPath(name).parts[0] == prefix}
        if not expected:
            continue
        # Direct/auto-expanded files remain the preferred compatibility path.
        direct_complete = all(any(
            _suffix_match(path, PurePosixPath(name))
            and path.stat().st_size == int(specs[name]["bytes"])
            and sha256_file(path) == specs[name]["sha256"]
            for path in input_files
        ) for name in expected)
        if direct_complete:
            continue
        archives = [path for path in input_files if path.name == archive_name]
        if len(archives) != 1:
            raise HybridFailure(f"retrieval_archive_not_unique:{archive_name}:{len(archives)}")
        archive_path = archives[0]
        mapped = _safe_archive_map(archive_path, prefix, expected)
        for name, info in mapped.items():
            if info.file_size != int(specs[name]["bytes"]):
                raise HybridFailure(f"retrieval_archive_member_size_mismatch:{name}")
        archive_members.update({name: (archive_path, info) for name, info in mapped.items()})
    for spec in manifest.get("files", []):
        relative = PurePosixPath(str(spec["path"]))
        candidates = [path for path in input_files if _suffix_match(path, relative)]
        candidates = [path for path in candidates if path.stat().st_size == int(spec["bytes"])]
        valid = [path for path in candidates if sha256_file(path) == spec["sha256"]]
        destination = stage.joinpath(*relative.parts)
        if valid:
            source = sorted(valid, key=lambda path: (len(path.parts), str(path)))[0]
            destination.parent.mkdir(parents=True, exist_ok=True)
            if not destination.exists():
                try:
                    os.link(source, destination)
                except OSError:
                    shutil.copy2(source, destination)
        elif relative.as_posix() in archive_members:
            archive_path, info = archive_members[relative.as_posix()]
            _copy_archive_member(
                archive_path, info, destination,
                int(spec["bytes"]), str(spec["sha256"]),
            )
        else:
            raise HybridFailure(f"retrieval_payload_file_missing:{relative}")
        if destination.stat().st_size != int(spec["bytes"]) or sha256_file(destination) != spec["sha256"]:
            raise HybridFailure(f"retrieval_stage_binding_failed:{relative}")
    return stage, manifest


def find_test_csv() -> Path:
    exact = INPUT_ROOT / "competitions/bengali-hallucination/test set.csv"
    if exact.is_file():
        return exact
    candidates = []
    competition_root = INPUT_ROOT / "competitions/bengali-hallucination"
    for path in competition_root.rglob("*.csv"):
        try:
            fields = set(pd.read_csv(path, nrows=1).columns)
        except Exception:
            continue
        if {"id", "prompt_bn", "response_bn", "context"}.issubset(fields):
            candidates.append(path)
    if len(candidates) != 1:
        raise HybridFailure(f"test_csv_not_unique:{len(candidates)}")
    return candidates[0]


def normalized_text(value: object) -> str:
    if value is None or pd.isna(value):
        return ""
    return str(value).replace("\r\n", "\n").replace("\r", "\n").strip()


def context_available(value: object) -> bool:
    return normalized_text(value).casefold() not in NULL_CONTEXTS


def prepare_rows(test_path: Path) -> tuple[list[dict[str, Any]], Path]:
    frame = pd.read_csv(test_path)
    required = ["id", "prompt_bn", "response_bn", "context"]
    if not set(required).issubset(frame.columns) or frame["id"].astype(str).duplicated().any():
        raise HybridFailure("competition_test_schema_or_id_failure")
    rows = []
    for source_index, raw in frame.iterrows():
        context = normalized_text(raw["context"])
        rows.append({
            "example_id": str(raw["id"]),
            "source_index": int(source_index),
            "model_prompt_bn": normalized_text(raw["prompt_bn"]),
            "model_response_bn": normalized_text(raw["response_bn"]),
            "model_context_bn": context,
            "context_available": context_available(raw["context"]),
            "formatting_provenance": {"status": "byte_preserving_newline_normalization_only"},
        })
    output = RUN_ROOT / "normalized_inputs.jsonl"
    atomic_text(output, "".join(canonical(row) + "\n" for row in rows))
    return rows, output


def load_lexical_records(stage: Path) -> list[dict[str, Any]]:
    path = stage / "artifacts/retrieval/phase2_lexical_seed_v2/lexical_seed_records.jsonl"
    return [json.loads(line) for line in path.read_text(encoding="utf-8-sig").splitlines() if line.strip()]


def compact_lexical_evidence(prompt: str, response: str, records: list[dict[str, Any]], limit: int = 3) -> list[dict[str, Any]]:
    folded_prompt = " ".join(prompt.casefold().split())
    folded_response = " ".join(response.casefold().split())
    operation = ""
    if "বিপরীত" in folded_prompt:
        operation = "antonym_lookup"
    elif "বাগধারা" in folded_prompt or "প্রবাদ" in folded_prompt:
        operation = "idiom_meaning_lookup"
    elif "উপসর্গ" in folded_prompt:
        operation = "prefix_origin_classification"
    if not operation:
        return []
    found = []
    for record in records:
        if record.get("operation") != operation or record.get("conflict_status") != "none":
            continue
        terms = [str(record.get("term_key", ""))] + [str(value) for value in record.get("display_terms", [])]
        if not any(term and " ".join(term.casefold().split()) in folded_prompt for term in terms):
            continue
        accepted = [" ".join(str(value).casefold().split()) for value in record.get("accepted_answers", [])]
        contrast = [" ".join(str(value).casefold().split()) for value in record.get("contrast_answers", [])]
        relation = "support_candidate" if folded_response in accepted else "counter_candidate" if folded_response in contrast else "neutral_candidate"
        found.append({
            "source_id": record.get("source_id"), "operation": operation,
            "term_key": record.get("term_key"), "accepted_answers": record.get("accepted_answers", []),
            "contrast_answers": record.get("contrast_answers", []), "evidence_role": relation,
        })
        if len(found) >= limit:
            break
    return found


def retrieval_map(path: Path) -> dict[str, dict[str, Any]]:
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
    return {str(row["example_id"]): row for row in rows}


TERMINAL_ELIGIBLE_EXACT_SOURCES = frozenset({
    "bangla_mmlu", "nctb_qa_87805", "nctb_education_aux",
})
_BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
_NUMERIC_COUNTER_GAP = re.compile(r"(?<=[0-9])\s+(?=(?:টি|টা|জন|খানা|খানি)(?:\s|$))")


def exact_answer_key(value: object) -> str:
    text = unicodedata.normalize("NFC", str(value or "")).translate(_BN_DIGITS)
    text = " ".join(text.casefold().split())
    return _NUMERIC_COUNTER_GAP.sub("", text)


def closed_exact_terminal(row: dict[str, Any], item: dict[str, Any]) -> dict[str, Any] | None:
    """Admit only conflict-free, operation/slot-aligned exact keyed evidence.

    Strict-v3 FTS, Wikipedia, WordNet, generated answers, fuzzy matches and
    containment are structurally unable to pass this gate.
    """

    if row.get("context_available") is True:
        return None
    merged = item.get("merged_source_candidate") or {}
    verdict = merged.get("verdict")
    if (
        verdict not in (0, 1)
        or merged.get("status") != "source_consensus_candidate"
        or merged.get("strict_exact_key_conflicts")
    ):
        return None
    response_key = exact_answer_key(row.get("model_response_bn"))
    if not response_key:
        return None
    candidates = list(item.get("retrieval_candidates") or [])
    qualified = []
    for candidate in candidates:
        source_verdict = candidate.get("source_verdict_candidate") or {}
        candidate_verdict = source_verdict.get("verdict")
        answer_keys = {exact_answer_key(value) for value in candidate.get("answers") or []}
        choice_keys = {exact_answer_key(value) for value in candidate.get("choices") or []}
        rule = str(source_verdict.get("rule") or "")
        aligned = (
            candidate.get("source_id") in TERMINAL_ELIGIBLE_EXACT_SOURCES
            and candidate.get("exact_normalized") is True
            and candidate.get("model_facing_eligible") is True
            and candidate.get("semantic_alignment_tier") == 0
            and candidate.get("slot_compatibility_tier") == 0
            and candidate.get("policy_compatibility_tier") in (0, 1)
            and candidate.get("number_set_match") is not False
            and candidate.get("negation_set_match") is not False
            and "containment" not in rule
        )
        exact_response_relation = (
            candidate_verdict == 1 and response_key in answer_keys
        ) or (
            candidate_verdict == 0
            and response_key in choice_keys
            and response_key not in answer_keys
        )
        if aligned and exact_response_relation:
            qualified.append(candidate)
    if not qualified or {int((value["source_verdict_candidate"])["verdict"]) for value in qualified} != {int(verdict)}:
        return None

    # An admitted exact candidate playing the opposite response-aware role is
    # a conflict even if it has no legacy terminal adapter.  Candidate order
    # is irrelevant: this is a set-style veto over the complete admitted pool.
    opposite_role = "counter_candidate" if int(verdict) == 1 else "support_candidate"
    for candidate in candidates:
        observed_role = candidate.get("evidence_role")
        if not observed_role:
            relation = str(candidate.get("response_answer_relation") or "none")
            observed_role = (
                "support_candidate"
                if relation == "exact"
                else (
                    "counter_candidate"
                    if candidate.get("exact_normalized") is True
                    and bool(candidate.get("answers"))
                    else "neutral_candidate"
                )
            )
        if (
            candidate not in qualified
            and candidate.get("exact_normalized") is True
            and candidate.get("model_facing_eligible") is True
            and candidate.get("semantic_alignment_tier") == 0
            and candidate.get("slot_compatibility_tier") == 0
            and candidate.get("policy_compatibility_tier") in (0, 1)
            and observed_role == opposite_role
        ):
            return None
    return {
        "verdict": int(verdict),
        "status": "closed_exact_key_terminal",
        "source_ids": sorted({str(value["source_id"]) for value in qualified}),
        "rules": sorted({str(value["source_verdict_candidate"]["rule"]) for value in qualified}),
        "exact_normalized": True,
        "conflict_free": True,
    }


def compact_retrieval(item: dict[str, Any], lexical: list[dict[str, Any]], limit: int = 8) -> str:
    lines = []
    for record in lexical:
        lines.append("LEXICAL_EXACT " + canonical(record))
    for candidate in list(item.get("retrieval_candidates") or [])[:limit]:
        compact = {
            "source_id": candidate.get("source_id"),
            "authority_class": candidate.get("authority_class"),
            "evidence_role": candidate.get("evidence_role"),
            "question": candidate.get("question"),
            "answers": candidate.get("answers"),
            "choices": candidate.get("choices"),
            "supporting_text": str(candidate.get("supporting_text") or "")[:480],
            "exact_normalized": candidate.get("exact_normalized"),
            "terminal_label_authority": candidate.get("terminal_label_authority"),
        }
        lines.append(canonical(compact))
    return "\n".join(lines) if lines else "[no admitted source candidate]"


CONTEXT_SYSTEM = 'তুমি বাংলা context-only factuality verifier; শুধু প্রদত্ত context ব্যবহার করবে। CONTEXT-ONLY VERIFICATION CONTRACT-এর ক্রম মেনে Recovered checks: question-context-response exact slot; Unicode NFC/joiner/digit; grammar theory/definition/rule; negation/quantifier/comparator/modality/clause scope; entity-relation-property; event phase; bounded math। প্রথমে question premise/answerability, তারপর exact entity-relation-property-answer type, event phase, time/date/number/unit এবং negation-quantifier-comparator-modality scope বাঁধবে। একই passage-এর অন্য question, slot, role, place, event বা date-এর answer গ্রহণ করবে না। Direct statement ছাড়াও কেবল supplied context-এর definition/theory/rule/formula এবং স্পষ্ট operands দিয়ে bounded math, age/duration, relative timeline, ordinal ও kinship relation করা যাবে। Grammar-এ exact operation/term/operands এবং conventional pair/register মানবে; context-এ theory/rule থাকলে তার সঠিক application supported হতে পারে। Unicode NFC, Bengali/ASCII digit, joiner বা attested OCR word-break comparison literal evidence বদলায় না এবং near-looking ভিন্ন শব্দকে সমান করে না। Partial containment কোনো unsupported extra claim ঢাকে না। বাইরের জ্ঞান, retrieval, web, lexical cache বা closed-book corpus নিষিদ্ধ। Ambiguous/conflicting/missing evidence, invalid premise, কোনো local window-এ তথ্য না থাকা refutation নয়; সব window-এর পরে unresolved conflict হলে not_enough_information। শুধু exact recognized antonym/idiom/prefix shell-এ hash-bound lexical table nonterminal evidence হতে পারে; exact operation, key, sense/register এবং conflict_status=none আবশ্যক। Fuzzy/generic corpus/Wikipedia নিষিদ্ধ। Samas/sandhi/affix/natva/satva supplied rule ও exact operands ছাড়া lookup করবে না। শুধু {"verdict":"supported"}, {"verdict":"refuted"}, অথবা {"verdict":"not_enough_information"} দাও।'
CLOSED_SYSTEM = 'তুমি বাংলা closed-book factuality verifier। নিচের offline retrieval কেবল nonterminal evidence; কোন hit, keyed answer, consensus candidate বা retrieval score নিজে verdict নয়। Retrieval miss মানে NEI, refutation নয়। প্রথমে exact requested operation, question slot, entity, relation/property, answer type, event/date, number/unit, negation এবং time/cultural scope মিলাও। তারপর সমানভাবে aligned support ও counter-evidence বিচার করো। Books/user OCR/BCS এবং curated sources Wikipedia-র উপরে, কিন্তু misaligned উচ্চ-authority evidence aligned নিম্ন-authority evidence-কে হারাতে পারবে না; Wikipedia corroboration-only। Antonym/idiom/prefix/sandhi/samas-এ exact operation, term, pair, sense ও register ছাড়া semantic near-match গ্রহণ করবে না। Evidence না থাকলে বা conflict unresolved হলে not_enough_information। শুধু {"verdict":"supported"}, {"verdict":"refuted"}, অথবা {"verdict":"not_enough_information"} দাও।'


def build_messages(row: dict[str, Any], retrieval: dict[str, Any], lexical_records: list[dict[str, Any]], router: Any) -> tuple[list[dict[str, str]], dict[str, Any]]:
    question = row["model_prompt_bn"]
    response = row["model_response_bn"]
    if row["context_available"]:
        user, route = build_contextual_policy_packet(
            row["model_context_bn"], question, response, router,
            lexical_records=lexical_records, max_windows=8, full_context_char_limit=6000,
        )
        return [{"role": "system", "content": CONTEXT_SYSTEM}, {"role": "user", "content": user}], route
    lexical = compact_lexical_evidence(question, response, lexical_records)
    evidence = compact_retrieval(retrieval, lexical)
    user = f"QUESTION:\n{question}\n\nADMITTED OFFLINE EVIDENCE:\n{evidence}\n\nRESPONSE TO VERIFY:\n{response}"
    return [{"role": "system", "content": CLOSED_SYSTEM}, {"role": "user", "content": user}], {"lexical_evidence": lexical}


def load_runtime(payload: Path, destination: Path) -> Path:
    with zipfile.ZipFile(payload) as archive:
        infos = archive.infolist()
        names = {info.filename for info in infos}
        if len(names) != len(infos) or "runtime_manifest.json" not in names:
            raise HybridFailure("runtime_member_set_invalid")
        manifest = json.loads(archive.read("runtime_manifest.json").decode("utf-8"))
        core = dict(manifest)
        manifest_id = core.pop("manifest_id", None)
        if manifest_id != RUNTIME_MANIFEST_ID or manifest_id != digest(core) or manifest.get("llama_cpp_commit") != LLAMA_COMMIT:
            raise HybridFailure("runtime_manifest_mismatch")
        declared = {row["file"]: row for row in manifest.get("files", [])}
        if set(declared) | {"runtime_manifest.json"} != names:
            raise HybridFailure("runtime_declared_members_mismatch")
        for info in infos:
            path = PurePosixPath(info.filename)
            mode = (info.external_attr >> 16) & 0xFFFF
            if info.is_dir() or path.is_absolute() or ".." in path.parts or "\\" in info.filename or stat.S_ISLNK(mode) or info.flag_bits & 1:
                raise HybridFailure("unsafe_runtime_member")
        archive.extractall(destination)
    binary = destination / "bin/llama-server"
    for path in (destination / "bin").iterdir():
        if path.is_file():
            path.chmod(0o755)
    if not binary.is_file() or not os.access(binary, os.X_OK):
        raise HybridFailure("llama_server_missing")
    return binary


def resolve_adapter_binding() -> tuple[Path, dict[str, Any]]:
    """Resolve one rights- and manifest-bound private adapter dataset root."""
    matches = []
    for manifest_path in INPUT_ROOT.rglob("artifact_manifest.json"):
        if not manifest_path.is_file() or manifest_path.is_symlink():
            continue
        try:
            manifest = read_json(manifest_path)
        except Exception:
            continue
        if manifest.get("dataset_id") == ADAPTER_DATASET_ID:
            matches.append((manifest_path.parent.resolve(), manifest))
    unique = {str(root): (root, manifest) for root, manifest in matches}
    if len(unique) != 1:
        raise HybridFailure(f"adapter_dataset_root_not_unique:{len(unique)}")
    root, manifest = next(iter(unique.values()))
    core = {key: value for key, value in manifest.items() if key != "manifest_id"}
    if (
        manifest.get("manifest_id") != ADAPTER_MANIFEST_ID
        or manifest.get("manifest_id") != digest(core)
        or manifest.get("private") is not True
        or manifest.get("private_competition_labels_present") is not False
        or manifest.get("lora_file") != ADAPTER_FILE
        or manifest.get("lora_bytes") != ADAPTER_BYTES
        or manifest.get("lora_sha256") != ADAPTER_SHA256
    ):
        raise HybridFailure("adapter_manifest_contract_mismatch")
    for name, expected in manifest.get("files", {}).items():
        path = root / name
        if (
            not path.is_file() or path.is_symlink()
            or path.stat().st_size != int(expected.get("bytes", -1))
            or sha256_file(path) != expected.get("sha256")
        ):
            raise HybridFailure(f"adapter_file_binding_mismatch:{name}")
    conversion = read_json(root / "conversion_receipt.json")
    conversion_core = {
        key: value for key, value in conversion.items()
        if key != "conversion_receipt_id"
    }
    if (
        conversion.get("conversion_receipt_id") != ADAPTER_CONVERSION_ID
        or conversion.get("conversion_receipt_id") != digest(conversion_core)
        or conversion.get("lora_gguf", {}).get("sha256") != ADAPTER_SHA256
        or conversion.get("llama_cpp_commit") != LLAMA_COMMIT
        or conversion.get("base_weights_read_by_converter") is not False
    ):
        raise HybridFailure("adapter_conversion_contract_mismatch")
    rights = read_json(root / "rights_record.json")
    rights_core = {key: value for key, value in rights.items() if key != "rights_record_id"}
    if (
        rights.get("rights_record_id") != ADAPTER_RIGHTS_ID
        or rights.get("rights_record_id") != digest(rights_core)
        or rights.get("visibility") != "private"
        or rights.get("private_competition_labels_used") is not False
        or rights.get("raw_model_reasoning_included") is not False
    ):
        raise HybridFailure("adapter_rights_contract_mismatch")
    adapter = root / ADAPTER_FILE
    if (
        adapter.stat().st_size != ADAPTER_BYTES
        or sha256_file(adapter) != ADAPTER_SHA256
    ):
        raise HybridFailure("adapter_gguf_binding_mismatch")
    return adapter, manifest


def server_command(binary: Path, model: Path, adapter: Path, port: int) -> list[str]:
    return [
        str(binary), "-m", str(model), "--lora", str(adapter), "-ngl", "99", "-c", "8192",
        "-b", "512", "-ub", "256", "-fa", "on", "--jinja",
        "--host", "127.0.0.1", "--port", str(port), "--no-webui",
    ]


def wait_healthy(process: subprocess.Popen[str], port: int, log_path: Path) -> float:
    started = time.perf_counter()
    for _ in range(600):
        if process.poll() is not None:
            raise HybridFailure(f"llama_server_exited:{process.returncode}:{log_path.read_text(encoding='utf-8', errors='replace')[-3000:]}")
        try:
            if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                return time.perf_counter() - started
        except requests.RequestException:
            pass
        time.sleep(1)
    raise HybridFailure("llama_server_health_timeout")


def topology() -> list[dict[str, Any]]:
    result = run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total", "--format=csv,noheader,nounits"], timeout=30)
    rows = []
    for line in result.stdout.splitlines():
        fields = [value.strip() for value in line.split(",")]
        rows.append({"index": int(fields[0]), "name": fields[1], "compute_capability": fields[2], "memory_mib": int(fields[3])})
    if len(rows) != 2 or any("T4" not in row["name"] for row in rows):
        raise HybridFailure(f"exact_t4x2_required:{rows}")
    return rows


def assign_queues(rows: list[dict[str, Any]]) -> list[list[dict[str, Any]]]:
    queues = [[], []]
    loads = [0, 0]
    for row in sorted(rows, key=lambda value: -(len(value["model_context_bn"]) + len(value["model_prompt_bn"]) + len(value["model_response_bn"]))):
        slot = 0 if loads[0] <= loads[1] else 1
        queues[slot].append(row)
        loads[slot] += len(row["model_context_bn"]) + len(row["model_prompt_bn"]) + len(row["model_response_bn"])
    return queues


def append_checkpoint(path: Path, rows: list[dict[str, Any]]) -> None:
    atomic_text(path, "".join(canonical(row) + "\n" for row in rows))


def run_queue(rows: list[dict[str, Any]], port: int, gpu_slot: int, retrieval: dict[str, dict[str, Any]], lexical_records: list[dict[str, Any]], router: Any, started: float) -> list[dict[str, Any]]:
    outputs = []
    checkpoint = RUN_ROOT / f"model_predictions_gpu{gpu_slot}.jsonl"
    for row in rows:
        row_started = time.perf_counter()
        item = retrieval[str(row["example_id"])]
        terminal = closed_exact_terminal(row, item)
        if terminal is not None:
            label = int(terminal["verdict"])
            method = "closed_exact_key_terminal"
            route_receipt = terminal
            finish_reason = "deterministic"
            prompt_tokens = completion_tokens = 0
        elif time.perf_counter() - started > DEADLINE_SECONDS:
            label = 0
            method = "deadline_fail_closed_nei"
            route_receipt = {"deadline": True}
            finish_reason = "deadline"
            prompt_tokens = completion_tokens = 0
        else:
            messages, route_receipt = build_messages(row, item, lexical_records, router)
            window_results = []
            if row["context_available"] and route_receipt.get("requires_window_aggregation"):
                window_calls = list(route_receipt.get("window_calls") or [])
                if not window_calls:
                    raise HybridFailure("context_window_plan_empty")
                for call in window_calls:
                    window_messages = [
                        {"role": "system", "content": CONTEXT_SYSTEM},
                        {"role": "user", "content": call["user"]},
                    ]
                    window_body = requests.post(
                        f"http://127.0.0.1:{port}/v1/chat/completions",
                        json={
                            "messages": window_messages, "temperature": 0.0, "top_p": 1.0,
                            "top_k": 40, "min_p": 0.05, "repeat_penalty": 1.0,
                            "seed": BASE_SEED + int(row["source_index"]) + int(call["window_index"]),
                            "max_tokens": MAX_NEW_TOKENS, "grammar": VERDICT_ONLY_GBNF,
                            "chat_template_kwargs": {"enable_thinking": False}, "stream": False,
                        }, timeout=900,
                    )
                    window_body.raise_for_status()
                    window_payload = window_body.json()
                    window_content = ((window_payload["choices"][0].get("message") or {}).get("content"))
                    if window_content not in VERDICT_ONLY_LITERALS:
                        window_content = '{"verdict":"not_enough_information"}'
                    window_results.append({
                        "window_index": call["window_index"],
                        "context_char_start": call["context_char_start"],
                        "context_char_end": call["context_char_end"],
                        "literal_span_sha256": call["literal_span_sha256"],
                        "literal_excerpt": call["literal_excerpt"],
                        "excerpt_char_start": call["excerpt_char_start"],
                        "excerpt_char_end": call["excerpt_char_end"],
                        "literal_excerpt_sha256": call["literal_excerpt_sha256"],
                        "window_verdict": json.loads(window_content)["verdict"],
                    })
                aggregate_user = build_aggregation_user(
                    row["model_prompt_bn"], row["model_response_bn"], window_results,
                    selected_notes=route_receipt.get("aggregation_selected_notes", []),
                    bounded_derivations=route_receipt.get("aggregation_bounded_derivations", []),
                    lexical_policy=route_receipt.get("contextual_lexical_policy"),
                )
                messages = [
                    {"role": "system", "content": CONTEXT_SYSTEM},
                    {"role": "user", "content": aggregate_user},
                ]
                route_receipt["window_result_ledger"] = window_results
                route_receipt["aggregation_prompt_sha256"] = hashlib.sha256(aggregate_user.encode("utf-8")).hexdigest()
                route_receipt["window_verdict_counts"] = {
                    verdict: sum(result["window_verdict"] == verdict for result in window_results)
                    for verdict in ("supported", "refuted", "not_enough_information")
                }
                route_receipt.pop("window_calls", None)
            body = requests.post(
                f"http://127.0.0.1:{port}/v1/chat/completions",
                json={
                    "messages": messages, "temperature": 0.0, "top_p": 1.0,
                    "top_k": 40, "min_p": 0.05, "repeat_penalty": 1.0,
                    "seed": BASE_SEED + int(row["source_index"]),
                    "max_tokens": MAX_NEW_TOKENS, "grammar": VERDICT_ONLY_GBNF,
                    "chat_template_kwargs": {"enable_thinking": False}, "stream": False,
                }, timeout=900,
            )
            body.raise_for_status()
            payload = body.json()
            choice = payload["choices"][0]
            content = (choice.get("message") or {}).get("content")
            parsed = VERDICT_ONLY_LITERALS.get(content)
            window_verdicts = {result["window_verdict"] for result in window_results}
            cross_window_conflict = {"supported", "refuted"} <= window_verdicts
            if cross_window_conflict:
                parsed = 0
                route_receipt["cross_window_conflict_forced_nei"] = True
                method = "cross_window_conflict_fail_closed_nei"
            else:
                method = "gbnf_verdict_only" if parsed in (0, 1) else "invalid_generation_fail_closed_nei"
            label = int(parsed) if parsed in (0, 1) else 0
            finish_reason = choice.get("finish_reason")
            usage = payload.get("usage") or {}
            prompt_tokens = usage.get("prompt_tokens")
            completion_tokens = usage.get("completion_tokens")
        output = {
            "example_id": row["example_id"], "source_index": row["source_index"],
            "label": label, "gpu_slot": gpu_slot, "method": method,
            "finish_reason": finish_reason, "context_available": row["context_available"],
            "latency_seconds": time.perf_counter() - row_started,
            "prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens,
            "route_receipt_sha256": hashlib.sha256(canonical(route_receipt).encode("utf-8")).hexdigest(),
            "raw_reasoning_persisted": False,
        }
        if row["context_available"]:
            output["context_diagnostic"] = {
                "policy_version": route_receipt.get("context_policy_version"),
                "policy_family_inventory_count": route_receipt.get("policy_family_inventory_count"),
                "canonical_policy_family_count": route_receipt.get("canonical_policy_family_count"),
                "engineered_evaluation_cell_count": route_receipt.get("engineered_evaluation_cell_count"),
                "operation_axis_count": route_receipt.get("operation_axis_count"),
                "canonical_policy_families": route_receipt.get("canonical_policy_families"),
                "engineered_evaluation_cells": route_receipt.get("engineered_evaluation_cells"),
                "operation_axis": route_receipt.get("operation_axis"),
                "detected_policy_families": route_receipt.get("detected_policy_families", []),
                "evidence_mode": route_receipt.get("evidence_mode"),
                "full_context_inference_coverage": route_receipt.get("full_context_inference_coverage"),
                "context_sha256": route_receipt.get("context_sha256"),
                "question_sha256": route_receipt.get("question_sha256"),
                "response_sha256": route_receipt.get("response_sha256"),
                "prompt_sha256": route_receipt.get("prompt_sha256"),
                "policy_receipt_sha256": route_receipt.get("receipt_sha256"),
                "external_retrieval_allowed": route_receipt.get("external_retrieval_allowed"),
                "window_count": route_receipt.get("window_count"),
                "window_verdict_counts": route_receipt.get("window_verdict_counts"),
                "cross_window_conflict_forced_nei": route_receipt.get("cross_window_conflict_forced_nei", False),
                "lexical_policy": route_receipt.get("contextual_lexical_policy"),
            }
        outputs.append(output)
        append_checkpoint(checkpoint, outputs)
    return outputs


def validate_and_write_submission(test_path: Path, outputs: list[dict[str, Any]]) -> Path:
    frame = pd.read_csv(test_path)
    by_id = {str(row["example_id"]): row for row in outputs}
    ids = [str(value) for value in frame["id"]]
    if len(by_id) != len(ids) or set(by_id) != set(ids):
        raise HybridFailure("prediction_id_coverage_failure")
    labels = [int(by_id[value]["label"]) for value in ids]
    if any(label not in (0, 1) for label in labels):
        raise HybridFailure("nonbinary_submission_label")
    destination = WORKING_ROOT / "submission.csv"
    temporary = destination.with_name("submission.csv.tmp")
    with temporary.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "label"])
        writer.writeheader()
        writer.writerows({"id": original, "label": label} for original, label in zip(frame["id"], labels, strict=True))
    os.replace(temporary, destination)
    check = pd.read_csv(destination)
    if list(check.columns) != ["id", "label"] or len(check) != len(frame) or not bool(check["label"].isin([0, 1]).all()):
        raise HybridFailure("written_submission_validation_failed")
    return destination


def main() -> int:
    started = time.perf_counter()
    for key, value in {
        "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1",
        "HF_HUB_DISABLE_TELEMETRY": "1", "CUDA_VISIBLE_DEVICES": "0,1",
        "NO_PROXY": "127.0.0.1,localhost", "PYTHONHASHSEED": "0",
    }.items():
        os.environ[key] = value
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    servers: list[tuple[subprocess.Popen[str], Any]] = []
    try:
        gpu_topology = topology()
        stage, retrieval_manifest = materialize_retrieval_payload()
        sys.path.insert(0, str(stage))
        from pipeline.contextual_note_taker_fallback import rank_context_windows
        from pipeline.phase2_sparse_retrieval import build as build_retrieval

        test_path = find_test_csv()
        rows, normalized_path = prepare_rows(test_path)
        retrieval_dir = RUN_ROOT / "retrieval"
        strict_composite = stage / "artifacts/retrieval/phase2_strict_rights_safe_fts_v3_final"
        if not (strict_composite / "manifest.json").is_file():
            raise HybridFailure("strict_v3_composite_cache_missing")
        retrieval_receipt = build_retrieval(
            normalized_path, retrieval_dir, top_k=8, batch_size=128,
            composite_cache_dir=strict_composite,
            composite_query_mode="all_closed",
        )
        composite_receipt = retrieval_receipt.get("composite_fts") or {}
        if (
            composite_receipt.get("enabled") is not True
            or composite_receipt.get("terminal_label_authority") is not False
            or composite_receipt.get("closed_book_only") is not True
        ):
            raise HybridFailure("strict_v3_composite_runtime_contract_failed")
        retrieved = retrieval_map(retrieval_dir / "retrieval.jsonl")
        lexical_records = load_lexical_records(stage)

        model = resolve_bound_file(MODEL_FILE, MODEL_BYTES, MODEL_SHA256)
        adapter, adapter_manifest = resolve_adapter_binding()
        runtime_payload = resolve_bound_file(RUNTIME_FILE, RUNTIME_BYTES, RUNTIME_SHA256)
        binary = load_runtime(runtime_payload, RUN_ROOT / "runtime")
        startup = []
        for slot, port in enumerate((8080, 8081)):
            log_path = RUN_ROOT / f"llama_server_gpu{slot}.log"
            handle = log_path.open("w", encoding="utf-8")
            environment = os.environ.copy()
            environment["CUDA_VISIBLE_DEVICES"] = str(slot)
            environment["LD_LIBRARY_PATH"] = os.pathsep.join(filter(None, (str(binary.parent), environment.get("LD_LIBRARY_PATH", ""))))
            process = subprocess.Popen(server_command(binary, model, adapter, port), stdout=handle, stderr=subprocess.STDOUT, text=True, env=environment)
            servers.append((process, handle))
            startup.append({"gpu_slot": slot, "seconds": wait_healthy(process, port, log_path)})

        queues = assign_queues(rows)
        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
            futures = [
                pool.submit(run_queue, queues[slot], 8080 + slot, slot, retrieved, lexical_records, rank_context_windows, started)
                for slot in range(2)
            ]
        outputs = [row for future in futures for row in future.result()]
        outputs.sort(key=lambda row: int(row["source_index"]))
        atomic_text(RUN_ROOT / "model_predictions.jsonl", "".join(canonical(row) + "\n" for row in outputs))
        submission = validate_and_write_submission(test_path, outputs)
        report = {
            "version": VERSION, "status": "complete_submission_not_submitted",
            "rows": len(outputs), "contextual_rows": sum(row["context_available"] for row in rows),
            "closed_book_rows": sum(not row["context_available"] for row in rows),
            "labels": dict(Counter(str(row["label"]) for row in outputs)),
            "methods": dict(Counter(str(row["method"]) for row in outputs)),
            "runtime_seconds": time.perf_counter() - started, "gpu_topology": gpu_topology,
            "server_startup": startup, "retrieval_manifest_id": retrieval_manifest["manifest_id"],
            "retrieval_runtime_manifest_sha256": sha256_file(retrieval_dir / "manifest.json"),
            "model": {
                "file": MODEL_FILE, "bytes": MODEL_BYTES, "sha256": MODEL_SHA256,
                "adapter_used": True, "adapter_dataset_id": ADAPTER_DATASET_ID,
                "adapter_file": ADAPTER_FILE, "adapter_bytes": ADAPTER_BYTES,
                "adapter_sha256": ADAPTER_SHA256,
                "adapter_manifest_id": adapter_manifest["manifest_id"],
                "adapter_server_argument": "--lora",
            },
            "submission": {"path": str(submission), "bytes": submission.stat().st_size, "sha256": sha256_file(submission)},
            "internet_required": False, "competition_submission_performed": False,
            "contextual_external_retrieval_allowed": False,
            "retrieval_terminal_label_authority": False,
            "retrieval_miss_means": "NEI_not_refutation",
            "closed_sources": "combined_private_v2_plus_strict_v3",
            "context_terminal_policy": "no_production_terminal_without_source_disjoint_held_admission",
            "context_policy_contract": "morichika-context-policy-v4-full-coverage-map-aggregate",
            "canonical_context_policy_families": 17,
            "engineered_context_evaluation_cells": 26,
            "context_operation_axis": 15,
            "current_affairs_included": False,
        }
        report["report_id"] = digest(report)
        atomic_json(RUN_ROOT / "run_report.json", report)
        print(json.dumps(report, ensure_ascii=False, indent=2))
        return 0
    except Exception as error:
        atomic_json(RUN_ROOT / "failure_report.json", {
            "version": VERSION, "status": "failed_closed", "error_type": type(error).__name__,
            "error": str(error), "elapsed_seconds": time.perf_counter() - started,
            "competition_submission_performed": False,
        })
        raise
    finally:
        for process, handle in servers:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    process.kill()
            handle.close()


if __name__ == "__main__":
    raise SystemExit(main())
